## Income Classification

### Objective

Build a model that can predict if a person's income is under or over $50,000. Various models will be built and compared - decision tree, random forest, logistic regression

### Data Dictionary

The data contains characteristics of the people

* age: age of a person 
* workclass: where person works 
* fnlwgt: weight of demographic characteristics - people with similar weights will have similar demographic characteristics
* education: education level of a person
* education-num: number of years of education a person has
* marital-status: marital status of the person
* occupation: occupation category of a person
* relationship : 
* race: race of the person
* sex: sex of the person
* capital-gain: investment gain of the person other than salary 
* capital-loss: loss from investments
* hours-per-week: number of hours a person works
* native-country: country the person lives in
* salary: >50K, <=50K (dependent variable, the salary is in Dollars per year)

### Loading Libraries

In [9]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

<h2>Data loading and Overview</h2>

In [137]:
df = pd.read_csv("Income.csv")

In [138]:
df.shape

(48842, 15)

<p style="font-size: 18px;font-weight: 500;">10-Row Sample:</p>

In [139]:
df.sample(10)

,age,workclass,fnlwgt,education,educational-num,marital-status,occupation,relationship,race,gender,capital-gain,capital-loss,hours-per-week,native-country,income
33284,53,Local-gov,205005,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,White,Female,0,0,60,United-States,>50K
45479,59,Private,314149,Assoc-voc,11,Married-civ-spouse,Sales,Husband,White,Male,0,1740,50,United-States,<=50K
11451,47,Private,46537,HS-grad,9,Never-married,Other-service,Not-in-family,White,Female,0,0,40,United-States,<=50K
18106,51,Private,221532,Bachelors,13,Divorced,Exec-managerial,Unmarried,White,Male,0,0,40,United-States,>50K
10062,56,Local-gov,238405,HS-grad,9,Widowed,Adm-clerical,Not-in-family,White,Female,0,0,40,United-States,<=50K
35442,23,Local-gov,314819,HS-grad,9,Never-married,Transport-moving,Own-child,White,Male,0,0,40,United-States,<=50K
14473,30,Private,259425,Assoc-voc,11,Never-married,Protective-serv,Not-in-family,White,Male,0,0,40,United-States,<=50K
27013,32,Self-emp-not-inc,134727,HS-grad,9,Married-civ-spouse,Machine-op-inspct,Husband,White,Male,0,0,40,United-States,<=50K
41977,31,Private,377374,Some-college,10,Divorced,Adm-clerical,Unmarried,Black,Female,0,0,40,Japan,<=50K
48089,53,Private,470368,Assoc-acdm,12,Divorced,Adm-clerical,Unmarried,White,Female,0,0,48,United-States,<=50K


<p>Null values are denoted using '?', rather than 'Null'.</p>

<p style="font-size: 18px;font-weight: 500;">Data types:</p>

In [140]:
df.dtypes

age                 int64
workclass          object
fnlwgt              int64
education          object
educational-num     int64
marital-status     object
occupation         object
relationship       object
race               object
gender             object
capital-gain        int64
capital-loss        int64
hours-per-week      int64
native-country     object
income             object
dtype: object

<p style="font-size: 18px;font-weight: 500;">General statistics:</p>

In [141]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 48842 entries, 0 to 48841
Data columns (total 15 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   age              48842 non-null  int64 
 1   workclass        48842 non-null  object
 2   fnlwgt           48842 non-null  int64 
 3   education        48842 non-null  object
 4   educational-num  48842 non-null  int64 
 5   marital-status   48842 non-null  object
 6   occupation       48842 non-null  object
 7   relationship     48842 non-null  object
 8   race             48842 non-null  object
 9   gender           48842 non-null  object
 10  capital-gain     48842 non-null  int64 
 11  capital-loss     48842 non-null  int64 
 12  hours-per-week   48842 non-null  int64 
 13  native-country   48842 non-null  object
 14  income           48842 non-null  object
dtypes: int64(6), object(9)
memory usage: 5.6+ MB


In [142]:
df.describe()

,age,fnlwgt,educational-num,capital-gain,capital-loss,hours-per-week
count,48842.000000,4.884200e+04,48842.000000,48842.000000,48842.000000,48842.000000
mean,38.643585,1.896641e+05,10.078089,1079.067626,87.502314,40.422382
std,13.710510,1.056040e+05,2.570973,7452.019058,403.004552,12.391444
min,17.000000,1.228500e+04,1.000000,0.000000,0.000000,1.000000
25%,28.000000,1.175505e+05,9.000000,0.000000,0.000000,40.000000
50%,37.000000,1.781445e+05,10.000000,0.000000,0.000000,40.000000
75%,48.000000,2.376420e+05,12.000000,0.000000,0.000000,45.000000
max,90.000000,1.490400e+06,16.000000,99999.000000,4356.000000,99.000000


<p style="font-size: 18px;font-weight: 500;">Finding missing values</p>

<p>Since there are many columns, and each will likely need to be handled separately, we are going to find missing values ahead of time, so it's easier to stay organized.</p>

<p style="font-size:16px; font-weight:500;">Total</p>

In [143]:
# Searching for the value "?" in all cells of the dataframe
df[(df.values.ravel() == "?").reshape(df.shape).any(1)].count()

age                3620
workclass          3620
fnlwgt             3620
education          3620
educational-num    3620
marital-status     3620
occupation         3620
relationship       3620
race               3620
gender             3620
capital-gain       3620
capital-loss       3620
hours-per-week     3620
native-country     3620
income             3620
dtype: int64

<p>As we can see, there are a total of 3620 rows with missing values.</p>

<p>However, many of these rows contain missing values in the 'occupation' and 'workclass' columns, which appear to be directly correlated. Let's see how many are accounted for in the total.</p>

In [144]:
df.loc[df['workclass'] == '?'].count()

age                2799
workclass          2799
fnlwgt             2799
education          2799
educational-num    2799
marital-status     2799
occupation         2799
relationship       2799
race               2799
gender             2799
capital-gain       2799
capital-loss       2799
hours-per-week     2799
native-country     2799
income             2799
dtype: int64

In [145]:
df.loc[df['occupation'] == '?'].count()

age                2809
workclass          2809
fnlwgt             2809
education          2809
educational-num    2809
marital-status     2809
occupation         2809
relationship       2809
race               2809
gender             2809
capital-gain       2809
capital-loss       2809
hours-per-week     2809
native-country     2809
income             2809
dtype: int64

<p>There are more rows with missing values for 'occupation' than there are for 'workclass', which means that there's at least one other value for 'workclass' which is associated with a missing value for occupation.</p>

<p>In order to find that value, i've created a new dataframe from only the rows that have a missing value for occupation.</p>

In [146]:
new_df = df[(df['occupation'] == '?')]

In [147]:
new_df.loc[new_df['workclass'] != '?'].sample(10)

,age,workclass,fnlwgt,education,educational-num,marital-status,occupation,relationship,race,gender,capital-gain,capital-loss,hours-per-week,native-country,income
39513,20,Never-worked,462294,Some-college,10,Never-married,?,Own-child,Black,Male,0,0,40,United-States,<=50K
13898,18,Never-worked,162908,11th,7,Never-married,?,Own-child,White,Male,0,0,35,United-States,<=50K
31053,17,Never-worked,237272,10th,6,Never-married,?,Own-child,White,Male,0,0,30,United-States,<=50K
21642,18,Never-worked,206359,10th,6,Never-married,?,Own-child,White,Male,0,0,40,United-States,<=50K
8785,17,Never-worked,131593,11th,7,Never-married,?,Own-child,Black,Female,0,0,20,United-States,<=50K
36618,18,Never-worked,157131,11th,7,Never-married,?,Own-child,White,Female,0,0,10,United-States,<=50K
48595,18,Never-worked,153663,Some-college,10,Never-married,?,Own-child,White,Male,0,0,4,United-States,<=50K
48585,30,Never-worked,176673,HS-grad,9,Married-civ-spouse,?,Wife,Black,Female,0,0,40,United-States,<=50K
11607,20,Never-worked,273905,HS-grad,9,Married-spouse-absent,?,Other-relative,White,Male,0,0,35,United-States,<=50K
27126,23,Never-worked,188535,7th-8th,4,Divorced,?,Not-in-family,White,Male,0,0,35,United-States,<=50K


<p style="font-size: 14px;">As shown, all 10 additional rows with missing values in occupation are associated with the value 'Never-worked' in workclass.</p>

<p>However, while we have 2809 rows accounted for, that still leaves 811 of the original 3620.</p>

In [148]:
# Iterate over each column, and print the number of rows with the value '?'
for column in df.columns.values:
    print(f"{column}: {(df[column].values == '?').sum()}")

age: 0
workclass: 2799
fnlwgt: 0
education: 0
educational-num: 0
marital-status: 0
occupation: 2809
relationship: 0
race: 0
gender: 0
capital-gain: 0
capital-loss: 0
hours-per-week: 0
native-country: 857
income: 0


<p style="font-size: 14px;">Now, it's clear that the remaining instances of missing values are for 'native-country'. Since there are more than 811, and none of the other columns have any missing values, this means there are 46 rows with missing values in all three.</p>

<h2>Data Preprocessing</h2>

Note: for this assignment, we are dropping fnlwgt, capital-gain, and capital-loss

In [149]:
df = df.drop(['fnlwgt'], axis=1)

In [150]:
df = df.drop(['capital-gain'], axis=1)

In [151]:
df = df.drop(['capital-loss'], axis=1)

### Duplicate Data

<p style="font-size: 14px;">The number of duplicate rows is insignificant, compared to the size of the dataframe, so they will be dropped for simplicity.</p>

In [155]:
df.shape

(41082, 12)

In [153]:
df = df.drop_duplicates()

In [154]:
df.shape

(41082, 12)

### Missing Values

<p style="font-size:14px;">Previously, there were:</p>
<ul>
    <li><b>2799</b> in workclass & occupation,</li>
    <li><b>10</b> in occupation & workclass = Never-worked</li>
    <li><b>811</b> in just native-country</li>
    <li><b>46</b> in all three</li>
</ul>

#### Workclass & Occupation

<p>Since there is a unique value for individuals who have never worked, I think it is safe to assume the remaining individuals with no occupation are those that have worked previously.</p>

<p>Their workclass will be changed to: <b>Not-working</b></p>
<p>Their occupation will be changed to: <b>Unemployed</b></p>

In [156]:
# Changing workclass value of aforementioned rows
df.loc[((df['workclass'] == '?') & (df['occupation'] == '?')), ['workclass']] = 'Not-working'

In [157]:
# Changing occupation value of aforementioned rows
df.loc[((df['workclass'] == 'Not-working') & (df['occupation'] == '?')), ['occupation']] = 'Unemployed'

<p>We also need to change the missing occupation value for individuals with a workclass of 'Never-worked'.</p>
<p>Their occupation will be changed to: <b>Unemployed</b></p>

In [158]:
# Changing occupation value of remaining missing values
df.loc[((df['workclass'] == 'Never-worked') & (df['occupation'] == '?')), ['occupation']] = 'Unemployed'

In [159]:
# Verifying
df.loc[((df['workclass'] == '?') | (df['occupation'] == '?'))]

,age,workclass,education,educational-num,marital-status,occupation,relationship,race,gender,hours-per-week,native-country,income


#### Native-Country

<p>Since this column isn't numerical, we could only ignore missing values if the column was insigificant to the objective of the dataset. However, native country almost certainly has a correlation to income, so rows with missing values here will be dropped.</p>

In [171]:
# Re-making the dataframe without rows containing a native-country value of '?'
df = df[df['native-country'] != '?']

### Outliers

In [160]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
age,41082.0,39.475780,13.864426,17.0,28.0,38.0,49.0,90.0
educational-num,41082.0,10.065722,2.666196,1.0,9.0,10.0,13.0,16.0
hours-per-week,41082.0,40.578380,12.992359,1.0,38.0,40.0,45.0,99.0


<p>All three of the numerical columns above are likely important to the objective of the dataset, and all minimum and maximum values for each could be justified.</p>

<p><b>For example:</b></p>
<p>Since years in education can vary significantly, depending on degree, field of study, and changes in career path, classifying any value at or below 16 as an outlier would be disingenuous.</p>
<p>There are plenty of occupations that work only parts of the year, but their volume of work during that period will be well above average (e.g. linemen during particularly stormy parts of the year)</p>
<p>It may not be exceedingly common for individuals in their 80s and 90s to be working, but it certainly happens. In highly specialized occupations this is even somewhat common.</p>

### Inconsistent Data

<p>All that needs to be done here is to identify and adjust any extraneous values left in non-numerical columns.</p>

In [177]:
df.dtypes

age                 int64
workclass          object
education          object
educational-num     int64
marital-status     object
occupation         object
relationship       object
race               object
gender             object
hours-per-week      int64
native-country     object
income             object
dtype: object

In [182]:
# Checking for unordinary values
df.workclass.value_counts()

workclass
Private             26560
Self-emp-not-inc     3612
Local-gov            2908
Not-working          2324
State-gov            1879
Self-emp-inc         1559
Federal-gov          1356
Without-pay            21
Never-worked           10
Name: count, dtype: int64

In [184]:
# Checking for unordinary values
df.education.value_counts()

education
HS-grad         12272
Some-college     8767
Bachelors        6351
Masters          2308
Assoc-voc        1903
11th             1563
Assoc-acdm       1531
10th             1245
7th-8th           899
Prof-school       748
9th               723
12th              595
Doctorate         533
5th-6th           477
1st-4th           234
Preschool          80
Name: count, dtype: int64

<p>Having separate values for each grade in HS and below seems somewhat unnecessary, so these values could be grouped in 'Some-HS' and 'No-HS', for example. However, there could be a difference in average income once hitting a certain grade (e.g. average incomes of 10th grade dropouts is lower than 11th grade dropouts), so for now it will remain unchanged.</p>

In [187]:
# Checking for unordinary values
df['marital-status'].value_counts()

marital-status
Married-civ-spouse       17722
Never-married            12819
Divorced                  6096
Separated                 1486
Widowed                   1475
Married-spouse-absent      594
Married-AF-spouse           37
Name: count, dtype: int64

In [193]:
# Checking for unordinary values
df.occupation.value_counts()

occupation
Prof-specialty       5278
Exec-managerial      5017
Adm-clerical         4643
Sales                4484
Craft-repair         4372
Other-service        4208
Machine-op-inspct    2357
Unemployed           2334
Transport-moving     1976
Handlers-cleaners    1681
Farming-fishing      1411
Tech-support         1298
Protective-serv       927
Priv-house-serv       229
Armed-Forces           14
Name: count, dtype: int64

In [194]:
# Checking for unordinary values
df.relationship.value_counts()

relationship
Husband           15245
Not-in-family     10997
Own-child          5641
Unmarried          4774
Wife               2154
Other-relative     1418
Name: count, dtype: int64

In [195]:
# Checking for unordinary values
df.race.value_counts()

race
White                 33606
Black                  4387
Asian-Pac-Islander     1385
Amer-Indian-Eskimo      469
Other                   382
Name: count, dtype: int64

In [196]:
# Checking for unordinary values
df.gender.value_counts()

gender
Male      26181
Female    14048
Name: count, dtype: int64

In [197]:
# Checking for unordinary values
df['native-country'].value_counts()

native-country
United-States                 36114
Mexico                          924
Philippines                     292
Germany                         205
Puerto-Rico                     184
Canada                          182
El-Salvador                     155
India                           151
Cuba                            138
England                         127
China                           122
South                           115
Jamaica                         105
Italy                           104
Dominican-Republic              102
Japan                            92
Poland                           87
Vietnam                          86
Guatemala                        86
Columbia                         85
Haiti                            75
Portugal                         67
Taiwan                           64
Iran                             59
Greece                           49
Nicaragua                        49
Peru                             46
Ecuador      